In [0]:
# 1. Cargar librerias inicales
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:

# 2. Leer bronze y pasar a pandas
df_silver = spark.table("workspace.weather_silver.weather")
df_silver.show(5, truncate=False)

In [0]:

# 3. Crear DIM_LOCATION
df_silver=df_silver.toPandas()
dim_location = (
    df_silver[
        [
            "department",
            "location_id",
            "latitude",
            "longitude",
            "timezone",
            "timezone_abbreviation",
            "elevation"
        ]
    ]
    .drop_duplicates()
)


# 4. Ordenar por departamento
dim_location = dim_location.sort_values(
    "department"
)

# 5. Ver resultado
print(dim_location.head())
print(dim_location.shape)

# 6. Pandas -> Spark
dim_location_spark = spark.createDataFrame(dim_location)
dim_location_spark.printSchema()

# 7. Guardar la dimension DIM_LOCATION
dim_location_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.weather_gold.dim_location")

In [0]:
%sql
select * from workspace.weather_gold.dim_location limit 5